# 公平对比入口（默认只做 CPU 合成）

这是后续 Colab 实验的入口。仓库里的 `colab_probe.py` 和 `colab_loader_test.py` 只是环境探测，不是本入口。`colab_stage_e_cont.py` 会从 ep10 往下续训，也不要拿来跑这四组。

默认不挂载 Drive、不训练、不下载权重或医学图像。把下面两个开关保持为 `False`，只跑合成 smoke。真实数据由协调方在自己的 Colab 里、确认 hostname 是同一台 VM 之后再开。

打开 `RUN_REAL` 也只会做 bf16 / 权重身份检查，然后停住。这一版没有训练循环，避免用错 head。

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "revision").exists():
    raise SystemExit("请把工作目录设为仓库根目录后再跑")
sys.path.insert(0, str(ROOT))
MOUNT_DRIVE = False
RUN_REAL = False
SEED = 7
SEED_PHASE = "single"  # 改成 triple 才会展开三组 seed
WEIGHT_ILS = ""  # 本地路径；留空则 fail-fast
WEIGHT_DINOV3 = ""
OUT_DIR = "/content/ils_fair_runs" if Path("/content").exists() else "/tmp/ils_fair_runs"
print("ROOT", ROOT)
print("OUT_DIR", OUT_DIR)


In [ ]:
import socket
import shutil
from pathlib import Path
print("hostname", socket.gethostname())
print("同一台 VM 以这个 hostname 为准，不要凭链接判断")
probe = Path(OUT_DIR)
probe.mkdir(parents=True, exist_ok=True)
usage = shutil.disk_usage(probe)
print("path", probe, "exists", probe.exists(), "disk_free_bytes", usage.free)


In [ ]:
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Drive 已按浏览器授权挂上。请再打印一次 hostname，确认和上面是同一台。")
else:
    print("未挂载 Drive。默认不访问私人数据。")


In [ ]:
from revision.preflight import environment_report
from revision.fair_config import runtime_plan, training_matrix
report = environment_report(OUT_DIR)
print(report)
print(runtime_plan(real=False, cuda=report["cuda"], bf16=report["bf16"]))
print("single-seed runs", len(training_matrix(phase=SEED_PHASE, seed=SEED)))


In [ ]:
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "revision.smoke", "--out", OUT_DIR])
print("smoke 日志", Path(OUT_DIR) / "smoke.log")


In [ ]:
from revision.identity import RevisionError, plan_resume, verify_identity
from revision.pretrain import resolve_backbone
sample = {
    "role": "fair_frozen_probe",
    "backbone_id": "innerlogic_surg_vitb16",
    "geometry": "square768",
    "epoch": 1,
    "best_val_dice": 0.0,
    "metric_name": "frame_equal_mean_gt_class_dice",
    "sha256": "example",
    "head_init": "random",
    "seed": SEED,
    "train_mode": "frozen_probe",
    "selection_split": "val",
    "continued_from_ep10": False,
    "epochs": 6,
}
print("resume", plan_resume(sample, sample))
blocked = dict(sample, epoch=10, continued_from_ep10=True)
try:
    plan_resume(blocked, sample)
except RevisionError as exc:
    print("ep10", exc.code)
for backbone_id, weight_path in (
    ("innerlogic_surg_vitb16", WEIGHT_ILS),
    ("dinov3_vitb16_lvd1689m", WEIGHT_DINOV3),
):
    try:
        resolve_backbone(backbone_id, weight_path or None)
    except RevisionError as exc:
        print(backbone_id, exc.code)
if RUN_REAL:
    runtime_plan(real=True, cuda=report["cuda"], bf16=report["bf16"])
    raise SystemExit("检查通过也不在这一版里开训。训练循环未包含。")
print("RUN_REAL 为 False，停在合成结果。日志和样例图在", OUT_DIR)


## 协调方以后要跑真实实验时

1. 浏览器里打开 Colab，第一格打印 hostname。挂载 Drive 的格子如果要开，先把 `MOUNT_DRIVE` 改成 True，挂完再打印一次 hostname，两次必须相同。
2. 权重用本地路径填进 `WEIGHT_ILS` / `WEIGHT_DINOV3`。InnerLogic 文件必须是 sha256 `a516a775…` 的 ViT-B/16。通用 DINOv3 在校验和写进注册表之前会 fail-fast，不能拿 InnerLogic 文件顶替。
3. 名为 `best_control.pt`、binary val 为 0.8682456 的文件是旧 B2，不是 Stage E control（0.9394298）。不要用它画最终 control。
4. 结果只写到新的 `fairrun-` 目录，并保留 `RUN_RECORD.json`。不要写进 `results_e` 或 `viz_ep10`。
5. 这一版没有训练循环。GPU 和 6 epoch 训练都还没跑。